In [ ]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [ ]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

In [ ]:
# === Global Configuration and Constants ===
start_sqrt_s = 101  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

lst_sigma_tot_born = []
lst_sqrt_s = []
lst_error = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0732

max_sqrt_s = 13000
step = 100
n_points = 10000

model_params = {
    'atlas': {
        'pl':  {'mg': 0.417, 'a1': 1.563, 'a2': 2.22}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}

lst_amp_born = []
lst_sqrt_s = []
lst_sigma_tot_born = []




In [ ]:
# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


# -------------------------------
# Inner integral (over phi)
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) - 
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n_points)
    return result

# -------------------------------
# Outer integral (over k)
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    return phi_integral(k, mg, a1, a2, m2_func, q, n_points)

# -------------------------------
# Double integral computation
# -------------------------------
def compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points=10000):
    result, _ = fixed_quad(
        lambda k: k_integral(k, mg, a1, a2, m2_func, q, n_points),
        0, sqrt_s_val, 
        n=n_points
    )
    return result

def born_amp(diff_T, s, epsilon, t):
    
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot_born(amp_born_value, s):
    return amp_born_value.imag / s * 0.389379323


In [ ]:

def calculate_born_cross_sections(start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points=10000):
    """Calculate cross sections for all sqrt_s values"""
    
    # Generate array of sqrt_s values
    sqrt_s_values = np.arange(start_sqrt_s, max_sqrt_s + step, step)
    
    # Process each sqrt_s value
    lst_sigma_tot_born = []
    lst_sqrt_s = []
    lst_amp_born = []
    
    for sqrt_s_val in sqrt_s_values:
        # Compute the double integral
        q = 0  
        diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points)
        
        # Calculate amplitude and cross section
        s = sqrt_s_val * sqrt_s_val
        amp_born_value = born_amp(diff_T, s, epsilon, 0)
        sigma_tot_born_value = sigma_tot_born(amp_born_value, s)
        
        # Store results
        lst_sigma_tot_born.append(sigma_tot_born_value)
        lst_sqrt_s.append(sqrt_s_val)
        lst_amp_born.append(amp_born_value)
    
    return lst_sigma_tot_born, lst_sqrt_s, lst_amp_born



In [ ]:
mass_model = 'pl'
ensemble = 'atlas'
m2_func = get_m2_function(mass_model)
params = model_params[ensemble][mass_model]
mg, a1, a2 = params['mg'], params['a1'], params['a2']
epsilon = epsilon_values[ensemble]


sqrt_s_values = np.arange(start_sqrt_s, max_sqrt_s + step, step)


# Calculate all cross sections
lst_sigma_tot_born, lst_sqrt_s, lst_amp_born = calculate_born_cross_sections(
    start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points
)

lst_s = [val**2 for val in lst_sqrt_s]

In [ ]:
from traceback import print_exception
import numpy as np
from scipy.integrate import fixed_quad
from scipy.special import j0

n = 2
q_max = 0.1
b_max = 10

# global lists
chi_list = []
amp_list = []

# -------------------------------
# Full chi(s,b) including k, phi, and new q integral
# -------------------------------
def chi(s, q, b, amp_born):
    return 1/s * q * j0(b * q) * amp_born

def eik_amp(s, b, q, chi):
    factor = 1 - np.exp(1j *chi)
    return 1j * s * b * j0(b * q) * factor 

# Example: loop over many b values
lst_b = np.linspace(0.1, 10, 10)
lst_q = np.linspace(0.01, 0.2, 10)
temp = []
lst_sqrt_s = np.linspace(100, 13000, 100)

for sqrt_s_val in lst_sqrt_s:
    eik_amp_sum = 0
    for b_val in lst_b:
        chi_sum = 0

        for q_val in lst_q:
            diff_t = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_pl, 0)
            born_amp_val = born_amp(diff_t, sqrt_s_val**2, epsilon, 0)
            chi_value = chi(sqrt_s_val**2, q_val, b_val, born_amp_val)
            chi_sum += chi_value
            # print(diff_t, born_amp_val, chi_value, chi_sum)

        eik_amp_value = eik_amp(sqrt_s_val**2, b_val, 0, chi_sum)
        eik_amp_sum += eik_amp_value
        print(eik_amp_value, eik_amp_sum)
        temp.append(eik_amp_sum.imag)


In [ ]:
# sum = 0
# temp_amp = []
# for i in sorted(temp):
#     sum += i 
#     temp_amp.append(sum)

In [ ]:
def add_iterative_curve(fig, x_data, y_data, 
                        curve_name:str=None, color:str='blue', line_type:str='lines+markers'):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode=line_type, 
    name=curve_name,
    line=dict(
        color=color,
        width=2),
    marker=dict(size=4))
)
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')

fig_amp = go.Figure()

add_iterative_curve(fig_amp, lst_sqrt_s, np.imag(lst_amp_born), curve_name='Im(A born)', color='blue')
add_iterative_curve(fig_amp, lst_sqrt_s, temp, curve_name='Im(A eikonal)', color='red')
fig_amp.update_layout(
    title='Amplitude Im vs. sqrt(s)',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Amp',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
fig_amp.show(renderer = 'browser')
